# village-seg-baseline — can a U-Net learn our six classes from one annotated village?

**Status:** `exploring`

The first hand-corrected export of `ua-village-annotated` is one location
(birky) on four dates, 3328 x 2560 at 0.385 m/px. This probe fine-tunes a
**U-Net / ResNet-34** on it and asks whether the classes are learnable from
this label set at all — on ground the model did not train on.

**What one village can and cannot say.** There is no second location, so this
measures fit to *birky*, not generalisation to villages. The hold-out is a
**spatial strip held out on all four dates** — dates share ground, so holding
out a date would leak. What it can settle: the pipeline works end to end, the
labels are consistent enough to learn from, and which initialisation to carry
forward.

**Two initialisations**, one per run (`init` parameter):

- `landcoverai` — the probe model from `pocs/landcoverai-finetune` (aerial,
  rural Poland, 5 classes). Its head rows are copied to our matching classes;
  `field` starts random.
- `imagenet` — the usual starting point, as the control for the above.

**Two baselines the model has to beat:**

1. The LandCover.ai probe **zero-shot**, run the way the annotator's adapter
   runs it. Caveat: `track` labels were *seeded* from this model, so its
   `track` IoU is inflated by construction.
2. **Label agreement across dates** — the ceiling for cross-date consistency
   of predictions.

`field` has **zero** labelled pixels in this export; it cannot be learned and
is left out of every mean.

Run from `src/` in the `uavloc` env:

`PYTORCH_ENABLE_MPS_FALLBACK=1 papermill pocs/village-seg-baseline/probe.ipynb pocs/village-seg-baseline/runs/probe-landcoverai.ipynb -p init landcoverai`

In [ ]:
# papermill parameters
SLUG = "village-seg-baseline"
location = "birky-49p9105-23p9192"
init = "landcoverai"   # "landcoverai" | "imagenet"
crop = 512
batch = 4
steps = 1200           # measured 2.46 s/step on MPS at b4/512 -> ~50 min
eval_every = 200
lr = 3e-4
rare_crop_p = 0.3      # share of crops centred on a pixel of a randomly chosen present class
holdout_frac = 0.25    # val strip: one quarter of the image, all dates
gap = 64               # px between val strip and any train crop
seed = 0

## Setup

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # must precede the torch import

import json, time, warnings
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import segmentation_models_pytorch as smp

warnings.filterwarnings("ignore")
assert Path("datasets").is_dir(), "run from src/ — cwd is " + str(Path.cwd())

from tools.annotator.refs._infer import tiled_logits   # same tiling the annotator's adapters use

DATA = Path("data/derived")
ANN = DATA / "ua-village-annotated"
HERE = Path("pocs") / SLUG
FIGS = HERE / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

CLASSES = ["background", "building", "track", "woody", "field", "water"]
SCORED = [0, 1, 2, 3, 5]        # field has no labels in this export
IGNORE = 255
DEV = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(seed); np.random.seed(seed)
print("torch", torch.__version__, "| smp", smp.__version__, "| device", DEV)

## Stage 0 — the data

Labels are already in each image's frame. A date aligned by `image_offset_px`
has a strip the shift uncovered, which export filled with background; that strip
is set to **ignore** here rather than taught as background.

In [ ]:
recs = [json.loads(l) for l in (ANN / "samples.jsonl").read_text().splitlines() if l.strip()]
recs = sorted((r for r in recs if r["location"] == location), key=lambda r: r["date"])


def uncovered(shape, off):
    """Pixels export filled because the date's shift uncovered them."""
    dx, dy = off
    h, w = shape
    m = np.zeros(shape, bool)
    if dx > 0: m[:, w - dx:] = True
    if dx < 0: m[:, :-dx] = True
    if dy > 0: m[h - dy:, :] = True
    if dy < 0: m[:-dy, :] = True
    return m


dates, images, labels = [], [], []
for r in recs:
    img = np.array(Image.open(DATA / r["image_file"]).convert("RGB"))
    lab = np.array(Image.open(ANN / r["label_file"]))
    assert img.shape[:2] == lab.shape, (img.shape, lab.shape)
    lab[uncovered(lab.shape, r["image_offset_px"])] = IGNORE
    dates.append(r["date"]); images.append(img); labels.append(lab)
H, W = labels[0].shape
MPP = recs[0]["mpp"]

print(f"{location}: {len(dates)} dates, {W}x{H} px at {MPP:.3f} m/px\n")
print(f"{'date':>12s} " + " ".join(f"{c:>10s}" for c in CLASSES) + f" {'ignored':>8s}")
for d, lab in zip(dates, labels):
    print(f"{d:>12s} " + " ".join(f"{(lab == i).mean():>10.2%}" for i in range(6))
          + f" {(lab == IGNORE).mean():>8.3%}")

### Pick the hold-out strip

Candidates: the four horizontal and four vertical quarters. Rule: every class
must keep most of its pixels for training (no class more than 35% held out),
and among those the strip holding out the largest share of the *worst-served*
class wins. Printed so the choice is checkable, not taken on trust.

In [ ]:
L = np.stack(labels)
tot = {c: (L == c).sum() for c in (1, 2, 3, 5)}
cands = []
for ax in ("h", "v"):
    n = int(round(1 / holdout_frac))
    for k in range(n):
        box = ((k * H // n, (k + 1) * H // n), (0, W)) if ax == "h" else ((0, H), (k * W // n, (k + 1) * W // n))
        (y0, y1), (x0, x1) = box
        share = {c: (L[:, y0:y1, x0:x1] == c).sum() / tot[c] for c in tot}
        cands.append((ax, k, box, share))
        print(f"{ax}{k}  " + "  ".join(f"{CLASSES[c]} {s:5.1%}" for c, s in share.items()))

ok = [c for c in cands if max(c[3].values()) <= 0.35]
ax, k, VAL, share = max(ok, key=lambda c: min(c[3].values()))
(vy0, vy1), (vx0, vx1) = VAL
print(f"\n-> hold-out {ax}{k}: rows {vy0}:{vy1}, cols {vx0}:{vx1}")

# Train crops may not touch the val strip or the gap around it.
banned = np.zeros((H, W), bool)
banned[max(0, vy0 - gap):vy1 + gap, max(0, vx0 - gap):vx1 + gap] = True

In [ ]:
fig, axes = plt.subplots(1, len(dates), figsize=(5 * len(dates), 4.2))
for a, d, img in zip(axes, dates, images):
    a.imshow(img[::4, ::4])
    a.add_patch(plt.Rectangle((vx0 / 4, vy0 / 4), (vx1 - vx0) / 4, (vy1 - vy0) / 4,
                              fill=False, ec="cyan", lw=2))
    a.set_title(f"{d}  (cyan = held out)", fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

## Stage 1 — model and sampler

Normalisation follows the initialisation: the LandCover.ai probe was trained on
`(x/255 - 0.5)/0.25`, the ImageNet encoder on ImageNet statistics.

Crops are 512 px (197 m of ground). With probability `rare_crop_p` a crop is
centred on a random pixel of a randomly chosen *present* class — without it,
water (0.2% of pixels) almost never appears in a batch.

In [ ]:
LCAI_OURS = [0, 1, 2, 3, 5]   # LandCover.ai probe output order -> our ids


def build_model(init):
    if init == "imagenet":
        return smp.Unet("resnet34", encoder_weights="imagenet", classes=len(CLASSES))
    if init == "landcoverai":
        m = smp.Unet("resnet34", encoder_weights=None, classes=len(CLASSES))
        src = torch.load("pocs/landcoverai-finetune/model.pt", map_location="cpu")
        dst = m.state_dict()
        for k, v in src.items():
            if k.startswith("segmentation_head"):
                dst[k][LCAI_OURS] = v          # copy matching class rows; field keeps random init
            else:
                dst[k] = v
        m.load_state_dict(dst)
        return m
    raise ValueError(init)


if init == "imagenet":
    MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)
else:
    MEAN, STD = np.full(3, 0.5, np.float32), np.full(3, 0.25, np.float32)


def norm(img):
    return (img.astype(np.float32) / 255.0 - MEAN) / STD


# Valid top-left corners: the whole crop must avoid the banned area.
ii = np.pad(banned.astype(np.int32).cumsum(0).cumsum(1), ((1, 0), (1, 0)))
hits = ii[crop:, crop:] - ii[:-crop, crop:] - ii[crop:, :-crop] + ii[:-crop, :-crop]
valid_tl = np.argwhere(hits == 0)
print(f"{len(valid_tl):,} valid crop positions")

# Per class, the train pixels a rare-centred crop may target.
centres = {}
for c in (1, 2, 3, 5):
    pts = []
    for di, lab in enumerate(labels):
        ys, xs = np.nonzero(lab == c)
        keep = hits[np.clip(ys - crop // 2, 0, H - crop), np.clip(xs - crop // 2, 0, W - crop)] == 0
        pts.append(np.c_[np.full(keep.sum(), di), ys[keep], xs[keep]])
    centres[c] = np.concatenate(pts)
    print(f"  {CLASSES[c]:>8s}: {len(centres[c]):>9,} train pixels to centre on")


def sample(rng):
    di = rng.integers(len(dates))
    if rng.random() < rare_crop_p:
        c = rng.choice([c for c in centres if len(centres[c])])
        di, y, x = centres[c][rng.integers(len(centres[c]))]
        t, l = np.clip(y - crop // 2, 0, H - crop), np.clip(x - crop // 2, 0, W - crop)
    else:
        t, l = valid_tl[rng.integers(len(valid_tl))]
    img = images[di][t:t + crop, l:l + crop]
    lab = labels[di][t:t + crop, l:l + crop]
    k = rng.integers(4)                                    # nadir: any rotation is legal
    img, lab = np.rot90(img, k), np.rot90(lab, k)
    if rng.random() < 0.5:
        img, lab = img[:, ::-1], lab[:, ::-1]
    x = norm(img)
    gain = 1 + rng.uniform(-0.15, 0.15) + rng.uniform(-0.05, 0.05, 3)   # brightness + per-channel tint
    x = (x * STD + MEAN) * gain
    x = (x - x.mean()) * (1 + rng.uniform(-0.15, 0.15)) + x.mean()      # contrast
    x = (x - MEAN) / STD
    return x.transpose(2, 0, 1).copy(), lab.copy()


def batch_of(rng, n):
    xs, ys = zip(*(sample(rng) for _ in range(n)))
    return torch.from_numpy(np.stack(xs)).float(), torch.from_numpy(np.stack(ys)).long()

In [ ]:
xb, yb = batch_of(np.random.default_rng(1), 8)
PAL = np.array([[40, 40, 40], [230, 30, 30], [255, 215, 0], [20, 160, 40],
                [200, 140, 255], [30, 110, 255]], np.uint8)


def colour(lab):
    out = np.zeros((*lab.shape, 3), np.uint8)
    ok = lab != IGNORE
    out[ok] = PAL[lab[ok]]
    out[~ok] = 255
    return out


fig, axes = plt.subplots(2, 8, figsize=(20, 5.2))
for i in range(8):
    im = xb[i].numpy().transpose(1, 2, 0) * STD + MEAN
    axes[0, i].imshow(np.clip(im, 0, 1)); axes[1, i].imshow(colour(yb[i].numpy()))
    for a in axes[:, i]: a.axis("off")
fig.suptitle("training crops as the model sees them (augmented) — sanity check, not evidence")
plt.tight_layout(); plt.show()

## Stage 2 — evaluation, defined before training

Per-class IoU on the hold-out strip, intersection and union **summed over all
four dates**, plus per date. `mIoU` = mean over background, building, track,
woody, water. Water in the strip is a few thousand pixels, so its IoU is noisy;
read it as a sign, not a number.

Cross-date consistency: over the strip, the share of pixels where the
prediction for date *i* equals the prediction for date *j*, averaged over all
pairs. The same quantity on the **labels** is the reference — the ground does
change between dates, so 100% is not the target.

In [ ]:
def strip(a):
    return a[vy0:vy1, vx0:vx1]


@torch.no_grad()
def predict(model, img, patch=crop, step=crop * 3 // 4):
    model.eval()
    lg = tiled_logits(model, norm(img), patch, step, len(CLASSES), DEV)
    return lg.argmax(0).astype(np.uint8)


def scores(preds, labs):
    """preds, labs: lists of strip arrays, one per date."""
    I = np.zeros(6); U = np.zeros(6); per_date = []
    for p, t in zip(preds, labs):
        ok = t != IGNORE
        i = np.array([((p == c) & (t == c) & ok).sum() for c in range(6)], float)
        u = np.array([(((p == c) | (t == c)) & ok).sum() for c in range(6)], float)
        I += i; U += u
        per_date.append({CLASSES[c]: (i[c] / u[c] if u[c] else None) for c in SCORED})
    iou = {CLASSES[c]: (I[c] / U[c] if U[c] else None) for c in SCORED}
    miou = float(np.mean([v for v in iou.values() if v is not None]))
    return {"iou": iou, "miou": miou, "per_date": per_date}


def agreement(arrs):
    pairs = [(i, j) for i in range(len(arrs)) for j in range(i + 1, len(arrs))]
    return float(np.mean([(arrs[i] == arrs[j]).mean() for i, j in pairs]))


val_labs = [strip(l) for l in labels]
print(f"label agreement across dates on the strip: {agreement(val_labs):.3f}")
print("val-strip pixels per class (all dates):",
      {CLASSES[c]: int(sum((v == c).sum() for v in val_labs)) for c in range(6)})

### Baseline 1 — the LandCover.ai probe, zero-shot

Run exactly as the annotator's `landcoverai` adapter runs it: resampled to its
0.25 m training scale, 256 px tiles. The image is predicted whole and then cut
to the strip, so edge context matches the fine-tuned model's.

In [ ]:
@torch.no_grad()
def lcai_zero_shot(img):
    m = smp.Unet("resnet34", encoder_weights=None, classes=5)
    m.load_state_dict(torch.load("pocs/landcoverai-finetune/model.pt", map_location="cpu"))
    m = m.eval().to(DEV)
    h0, w0 = img.shape[:2]
    s = MPP / 0.25
    im = np.asarray(Image.fromarray(img).resize((int(w0 * s), int(h0 * s)), Image.BICUBIC))
    x = (im.astype(np.float32) / 255.0 - 0.5) / 0.25
    p = tiled_logits(m, x, 256, 256, 5, DEV).argmax(0).astype(np.uint8)
    p = np.array(Image.fromarray(p).resize((w0, h0), Image.NEAREST))
    return np.array(LCAI_OURS, np.uint8)[p]


t0 = time.time()
zs_preds = [strip(lcai_zero_shot(img)) for img in images]
zero_shot = scores(zs_preds, val_labs)
zero_shot["agreement"] = agreement(zs_preds)
print(f"zero-shot ({time.time() - t0:.0f} s)  mIoU {zero_shot['miou']:.3f}  "
      f"agreement {zero_shot['agreement']:.3f}")
print({k: round(v, 3) for k, v in zero_shot["iou"].items() if v is not None})

## Stage 3 — train

CE (ignore index 255) + soft Dice over the classes present in the batch. AdamW
with cosine decay. Evaluated on the strip every `eval_every` steps. The **last**
checkpoint is the one reported, not the best one: the strip is the only held-out
data, and choosing the best step on it would score the model on the data used
to pick it.

In [ ]:
def dice_loss(logits, y):
    ok = (y != IGNORE)
    p = logits.softmax(1) * ok[:, None]
    oh = F.one_hot(y.clamp(max=len(CLASSES) - 1), len(CLASSES)).permute(0, 3, 1, 2) * ok[:, None]
    present = oh.sum((0, 2, 3)) > 0
    inter = (p * oh).sum((0, 2, 3)); den = p.sum((0, 2, 3)) + oh.sum((0, 2, 3))
    d = 1 - (2 * inter + 1) / (den + 1)
    return d[present].mean()


model = build_model(init).to(DEV)
opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.05)
rng = np.random.default_rng(seed)

s0 = scores([strip(predict(model, img)) for img in images], val_labs)
history = [{"step": 0, "loss": None, "miou": s0["miou"], "iou": s0["iou"]}]
print(f"step    0  mIoU {s0['miou']:.3f}  (initialisation, before any training)")

t0, run_loss = time.time(), []
for step in range(1, steps + 1):
    model.train()
    x, y = batch_of(rng, batch)
    x, y = x.to(DEV), y.to(DEV)
    out = model(x)
    loss = F.cross_entropy(out, y, ignore_index=IGNORE) + dice_loss(out, y)
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    run_loss.append(loss.item())
    if step % eval_every == 0 or step == steps:
        s = scores([strip(predict(model, img)) for img in images], val_labs)
        history.append({"step": step, "loss": float(np.mean(run_loss)), "miou": s["miou"], "iou": s["iou"]})
        el = time.time() - t0
        print(f"step {step:4d}  loss {np.mean(run_loss):.3f}  mIoU {s['miou']:.3f}  "
              + " ".join(f"{k[:4]} {v:.2f}" for k, v in s["iou"].items() if v is not None)
              + f"  [{el / 60:.0f} min, {el / step:.2f} s/step]")
        run_loss = []

train_min = (time.time() - t0) / 60
torch.save(model.state_dict(), HERE / f"model-{init}.pt")

## Stage 4 — numbers

In [ ]:
preds = [predict(model, img) for img in images]
val_preds = [strip(p) for p in preds]
final = scores(val_preds, val_labs)
final["agreement"] = agreement(val_preds)

print(f"{'class':>10s} {'zero-shot':>10s} {'fine-tuned':>11s}")
for c in SCORED:
    n = CLASSES[c]
    z, f = zero_shot["iou"][n], final["iou"][n]
    print(f"{n:>10s} {z:>10.3f} {f:>11.3f}")
print(f"{'mIoU':>10s} {zero_shot['miou']:>10.3f} {final['miou']:>11.3f}")
print(f"\ncross-date agreement on strip: labels {agreement(val_labs):.3f} | "
      f"zero-shot {zero_shot['agreement']:.3f} | fine-tuned {final['agreement']:.3f}")

print("\nper date (fine-tuned):")
for d, pd_ in zip(dates, final["per_date"]):
    print(f"  {d:>12s} " + " ".join(f"{k[:4]} {v:.2f}" if v is not None else f"{k[:4]}  -  "
                                    for k, v in pd_.items()))

# Confusion on the strip, all dates, rows = label, cols = prediction (row-normalised)
cm = np.zeros((6, 6))
for p, t in zip(val_preds, val_labs):
    ok = t != IGNORE
    np.add.at(cm, (t[ok], p[ok]), 1)
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
print("\nconfusion (row = label, col = prediction), row-normalised:")
print(f"{'':>10s} " + " ".join(f"{c[:6]:>7s}" for c in CLASSES))
for i, c in enumerate(CLASSES):
    print(f"{c:>10s} " + " ".join(f"{v:>7.2f}" for v in cmn[i]))

results = {"location": location, "init": init, "val_box": VAL, "dates": dates,
           "params": {"crop": crop, "batch": batch, "steps": steps, "lr": lr,
                      "rare_crop_p": rare_crop_p, "seed": seed, "device": str(DEV)},
           "train_minutes": train_min, "zero_shot": zero_shot, "final": final,
           "label_agreement": agreement(val_labs), "history": history,
           "confusion": cm.tolist()}
(HERE / f"results-{init}.json").write_text(json.dumps(results, indent=1, default=float))

## Look at it

In [ ]:
steps_ = [h["step"] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(steps_[1:], [h["loss"] for h in history[1:]], "k.-"); axes[0].set_title("train loss (mean per interval)")
for c in SCORED:
    axes[1].plot(steps_, [h["iou"][CLASSES[c]] for h in history], ".-", color=PAL[c] / 255, label=CLASSES[c])
axes[1].plot(steps_, [h["miou"] for h in history], "k--", lw=2, label="mIoU")
axes[1].set_title("hold-out strip IoU"); axes[1].legend(fontsize=8); axes[1].set_ylim(0, 1)
for a in axes: a.set_xlabel("step"); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIGS / f"{init}-curve.png", dpi=110); plt.show()

In [ ]:
# Whole hold-out strip, every date: image / label / prediction / errors (red = wrong)
rows = []
for d, img, t, p in zip(dates, images, val_labs, val_preds):
    err = np.where((t != IGNORE) & (p != t), 1, 0)
    e = strip(img).copy(); e[err == 1] = [255, 0, 0]
    rows += [(f"{d} image", strip(img)), (f"{d} label", colour(t)),
             (f"{d} prediction", colour(p)), (f"{d} errors", e)]
aspect = (vy1 - vy0) / (vx1 - vx0)
fig, axes = plt.subplots(len(rows), 1, figsize=(18, 18 * aspect * len(rows) + 1))
for a, (title, im) in zip(axes, rows):
    a.imshow(im); a.set_title(title, fontsize=9, loc="left"); a.axis("off")
plt.tight_layout(); plt.savefig(FIGS / f"{init}-holdout.png", dpi=90, bbox_inches="tight"); plt.show()

In [ ]:
# Native-resolution zoom: the densest-building window of the strip, all dates
bm = sum((v == 1).astype(np.int32) for v in val_labs)
z = min(400, vy1 - vy0, vx1 - vx0)
k_ = np.ones(z) / z
dens = np.apply_along_axis(lambda r: np.convolve(r, k_, "valid"), 1, bm)
dens = np.apply_along_axis(lambda c_: np.convolve(c_, k_, "valid"), 0, dens)
zy, zx = np.unravel_index(dens.argmax(), dens.shape)
sl = (slice(zy, zy + z), slice(zx, zx + z))

fig, axes = plt.subplots(3, len(dates), figsize=(4.5 * len(dates), 13.5))
for j, (d, img, t, p) in enumerate(zip(dates, images, val_labs, val_preds)):
    axes[0, j].imshow(strip(img)[sl]); axes[0, j].set_title(d, fontsize=9)
    axes[1, j].imshow(colour(t[sl])); axes[2, j].imshow(colour(p[sl]))
axes[0, 0].set_ylabel("image"); axes[1, 0].set_ylabel("label"); axes[2, 0].set_ylabel("prediction")
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
handles = [plt.Rectangle((0, 0), 1, 1, color=PAL[c] / 255) for c in range(6)]
fig.legend(handles, CLASSES, loc="lower center", ncol=6)
plt.tight_layout(rect=(0, .03, 1, 1)); plt.savefig(FIGS / f"{init}-zoom.png", dpi=100); plt.show()

## Verdict

Filled in `notes.md` after both runs (`landcoverai`, `imagenet`) — the verdict
compares them, so it does not belong to either executed copy.